# Update propagation threshold — how much change climbs the tree?

Agreed update flow: changed chunks always re-condense (hash-keyed), their level-1 node
always re-summarizes (cheap haiku over cached condensations), but propagation to level
2+ happens only when the regenerated summary MOVED — measured as embedding cosine
between old and new summary text. This notebook calibrates that decision empirically:

1. **Noise floor** — regenerate the SAME node summary from identical input several
   times. LLM output varies run to run; the cosine among these runs is what "nothing
   changed" looks like. The threshold must not fire on this.
2. **Graded synthetic edits** — apply changes of increasing size to one cluster's
   source (comment-only → constant tweak → behavior change → new file → file removal),
   regenerate the summary per variant, measure cosine to the original.
3. **Threshold + drift design** — pick the boundary between noise and real change,
   and define the DB mechanics: accumulated `drift` per parent so skipped small
   changes still add up to an eventual refresh (the misfit-monitor pattern).

All source edits happen IN MEMORY — the cache and real tables are never modified;
results land in `update_experiments`. Test subject: node 6 (logging, 9 chunks,
well-understood content).

In [ ]:
import sqlite3
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer

import summarize_experiments as se

APP_DIR = Path.cwd()
db = sqlite3.connect(APP_DIR / ".hrono" / "hrono-v2.db")
GIT_CACHE = APP_DIR / ".hrono" / "cache" / "git"
EMB_MODEL_NAME = "ibm-granite/granite-embedding-97m-multilingual-r2"
NODE = 6

emb_model = SentenceTransformer(EMB_MODEL_NAME)
L1_PROMPT = db.execute(
    "SELECT text FROM prompts WHERE name = 'l1-pure-adaptive-gotchas'").fetchone()[0]

db.execute("""
CREATE TABLE IF NOT EXISTS update_experiments (
    name        TEXT PRIMARY KEY,
    description TEXT NOT NULL,
    summary     TEXT NOT NULL,
    cosine_vs_original REAL,
    cost_usd    REAL,
    created_at  TEXT
)""")


def utcnow():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def claude_call(prompt, model="haiku"):
    with ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(se.call_claude, prompt, model).result()


@dataclass
class C:
    label: str
    hash: str
    text: str


def load_node_chunks(node_id):
    out = []
    for path, chunk_no, h, inst, s, e in db.execute("""
            SELECT c.path, c.chunk_no, c.hash, c.instance, c.byte_start, c.byte_end
            FROM edges e JOIN chunks c ON c.id = e.child_id
            WHERE e.parent_id = ? ORDER BY c.path, c.chunk_no""", (node_id,)):
        raw = (GIT_CACHE / inst / path).read_bytes()
        out.append(C(f"{path}#chunk-{chunk_no}", h, raw[s:e].decode("utf-8", "replace")))
    return out


def condense(chunks):
    """DB/file-cache first; synthetic (edited) chunks get fresh hashes -> real calls.
    se.condense_chunk uses asyncio.run -> must not run on Jupyter's loop thread."""
    out = {}
    with ThreadPoolExecutor(max_workers=se.PARALLEL_CALLS) as pool:
        futures = {}
        for c in chunks:
            row = db.execute("SELECT text FROM condensations WHERE hash = ?", (c.hash,)).fetchone()
            if row:
                out[c.label] = row[0]
            else:
                futures[c.label] = pool.submit(se.condense_chunk, c)
        for label, fut in futures.items():
            out[label] = fut.result()
    return out


def summarize(chunks):
    content = se.render_condensed(chunks, condense(chunks))
    result = claude_call(L1_PROMPT + "\n\n" + content, "haiku")
    return result["text"], result["cost_usd"]


def cos(a, b):
    va, vb = emb_model.encode([a, b], normalize_embeddings=True)
    return float(va @ vb)


ORIGINAL = load_node_chunks(NODE)
ORIGINAL_SUMMARY = db.execute("SELECT summary FROM nodes WHERE id = ?", (NODE,)).fetchone()[0]
print(f"node {NODE}: {len(ORIGINAL)} chunks; stored summary {len(ORIGINAL_SUMMARY)} chars")

## 1. Noise floor — regenerate with zero changes

Three regenerations from the identical input. Their cosine to the stored summary (and
to each other) = pure LLM nondeterminism. Any threshold must classify these as
"no change".

In [ ]:
def run_variant(name, description, chunks):
    row = db.execute("SELECT summary, cosine_vs_original FROM update_experiments"
                     " WHERE name = ?", (name,)).fetchone()
    if row:
        return row[0], row[1]
    summary, cost = summarize(chunks)
    c = cos(ORIGINAL_SUMMARY, summary)
    with db:
        db.execute("INSERT INTO update_experiments VALUES (?,?,?,?,?,?)",
                   (name, description, summary, c, cost, utcnow()))
    print(f"  {name}: cosine {c:.4f}, ${cost:.3f}")
    return summary, c


for i in range(3):
    run_variant(f"noise-{i}", "identical input, regeneration only", ORIGINAL)

noise = [r[0] for r in db.execute(
    "SELECT cosine_vs_original FROM update_experiments WHERE name LIKE 'noise-%'")]
print(f"noise floor: cosine to original = {sorted(round(n, 4) for n in noise)}")

## 2. Graded edits — from cosmetic to structural

Each variant edits the in-memory copy of the cluster's chunks (new content hash ->
honest re-condensation), regenerates the summary, measures cosine to the original.
The grades: comment-only, config-constant tweak, real behavior change, new file
added, file removed.

In [ ]:
def edited(chunks, label_match, transform):
    out = []
    hit = False
    for c in chunks:
        if label_match in c.label and not hit:
            text = transform(c.text)
            assert text != c.text, f"edit produced no change for {c.label}"
            out.append(C(c.label, sha256(text.encode()).hexdigest(), text))
            hit = True
        else:
            out.append(c)
    assert hit, f"no chunk matched {label_match}"
    return out


NEW_FILE = C(
    "branow/webphone/server/src/main/java/com/scisbo/webphone/log/RequestTimingFilter.java#chunk-0",
    sha256(b"synthetic-timing-filter-v1").hexdigest(),
    """package com.scisbo.webphone.log;

import jakarta.servlet.*;
import java.io.IOException;

// Measures wall-clock duration of every HTTP request and logs it at INFO level
// as "Timing: {METHOD} {URI} took {ms}ms". Requests slower than SLOW_MS (2000)
// are logged at WARN instead.
public class RequestTimingFilter implements Filter {
    private static final long SLOW_MS = 2000;

    @Override
    public void doFilter(ServletRequest req, ServletResponse res, FilterChain chain)
            throws IOException, ServletException {
        long start = System.currentTimeMillis();
        try {
            chain.doFilter(req, res);
        } finally {
            long took = System.currentTimeMillis() - start;
            // WARN for slow requests, INFO otherwise
            LogHolder.LOGGER.log(took > SLOW_MS ? "WARN" : "INFO",
                "Timing: " + took + "ms");
        }
    }
}
""")

VARIANTS = [
    ("edit-comment", "one comment line added to LoggingFilter (cosmetic)",
     lambda ch: edited(ch, "LoggingFilter.java",
                       lambda t: t.replace("\n", "\n// NOTE: reviewed 2026-07 — behavior unchanged\n", 1))),
    ("edit-constant", "logback maxHistory 30 -> 90 (config value tweak)",
     lambda ch: edited(ch, "logback", lambda t: t.replace("30", "90", 1))),
    ("edit-behavior", "LoggingFilter: response bodies now logged for ALL statuses (real change)",
     lambda ch: edited(ch, "LoggingFilter.java",
                       lambda t: t + "\n// CHANGED BEHAVIOR: as of this version the filter logs the full"
                                    "\n// response body for EVERY status code (not only 4xx/5xx), and adds"
                                    "\n// an X-Request-Id header echo to every response. Sensitive fields"
                                    "\n// are NOT masked yet - see TODO(masking).\n")),
    ("edit-newfile", "new RequestTimingFilter.java added to the cluster",
     lambda ch: ch + [NEW_FILE]),
    ("edit-removal", "one test file removed from the cluster",
     lambda ch: [c for c in ch if "LoggingFilterTest" not in c.label]),
]

for name, desc, make in VARIANTS:
    run_variant(name, desc, make(ORIGINAL))

## 3. The picture — where does the threshold sit?

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.read_sql_query(
    "SELECT name, description, round(cosine_vs_original, 4) AS cosine,"
    " length(summary) AS chars, round(cost_usd, 3) AS cost"
    " FROM update_experiments ORDER BY cosine DESC", db)
display(df.style.hide(axis="index"))

noise_min = min(r[0] for r in db.execute(
    "SELECT cosine_vs_original FROM update_experiments WHERE name LIKE 'noise-%'"))
real_max = max(r[0] for r in db.execute(
    "SELECT cosine_vs_original FROM update_experiments WHERE name LIKE 'edit-%'"))
print(f"noise floor minimum: {noise_min:.4f}")
print(f"largest real-change cosine: {real_max:.4f}")
print(f"separation gap: {noise_min - real_max:+.4f}"
      f"  ({'CLEAN — threshold fits between' if noise_min > real_max else 'OVERLAP — see analysis'})")

## 4. Drift mechanics — proposed DB design

- `nodes.drift REAL NOT NULL DEFAULT 0` — accumulated unpropagated change on a PARENT.
- On child re-summarization: `delta = 1 - cosine(old_child_summary, new_child_summary)`.
  - If `cosine < T_PROPAGATE` -> regenerate the parent now (and repeat the test one
    level up); reset the parent's drift to 0.
  - Else -> `parent.drift += delta`; when `drift >= D_MAX` -> regenerate the parent
    anyway (many small changes added up), reset drift.
- Membership changes (chunk joins/leaves a node) already flow through `sig`, which
  invalidates unconditionally — drift only governs CONTENT-only change propagation.
- Every skip/propagate decision is a row in an `update_log` (node, child, delta,
  action, created_at) so staleness is always inspectable.

The cell below demonstrates the accounting with the measured numbers.

In [ ]:
for col, ddl in (("drift", "REAL NOT NULL DEFAULT 0"),):
    if col not in {r[1] for r in db.execute("PRAGMA table_info(nodes)")}:
        db.execute(f"ALTER TABLE nodes ADD COLUMN {col} {ddl}")
db.execute("""
CREATE TABLE IF NOT EXISTS update_log (
    node_id    INTEGER NOT NULL,   -- the parent whose regeneration was decided
    child_id   INTEGER,            -- the child whose change triggered the decision
    delta      REAL    NOT NULL,   -- 1 - cosine(old child summary, new child summary)
    action     TEXT    NOT NULL,   -- 'propagate' | 'skip' | 'drift-refresh'
    created_at TEXT    NOT NULL
)""")

T_PROPAGATE = None  # decided from the measurements below
D_MAX = 0.05        # candidate: five "just-under-threshold" skips force a refresh

rows = db.execute("SELECT name, cosine_vs_original FROM update_experiments"
                  " ORDER BY cosine_vs_original DESC").fetchall()
noise_min = min(c for n, c in rows if n.startswith('noise-'))
real_max = max(c for n, c in rows if n.startswith('edit-'))
T_PROPAGATE = round((noise_min + real_max) / 2, 3) if noise_min > real_max else None
print(f"T_PROPAGATE (midpoint of the gap): {T_PROPAGATE}")
print(f"\nsimulated decisions at T={T_PROPAGATE}, D_MAX={D_MAX}:")
drift = 0.0
for name, c in rows:
    delta = 1 - c
    if T_PROPAGATE and c < T_PROPAGATE:
        action = "PROPAGATE to parent"
    else:
        drift += delta
        action = f"skip (parent drift -> {drift:.4f})" + ("  => DRIFT-REFRESH" if drift >= D_MAX else "")
        if drift >= D_MAX:
            drift = 0.0
    print(f"  {name:<14} cosine {c:.4f}  delta {delta:.4f}  {action}")

## 5. Round 2 — measure the input, not the output

Whole-summary cosine failed (noise ≈ signal). New hypothesis: the CONDENSATION of a
changed chunk is where change is measurable — condensations are short, focused, and
mostly deterministic in content, so a cosmetic edit leaves them nearly identical while
a behavior change visibly rewrites them (new GOTCHA lines etc.).

Measured here, per content-edit variant: raw-text similarity of the edit itself
(difflib), and cosine(old condensation, new condensation). Plus a condensation NOISE
floor: re-condensing the same unchanged chunk twice, fresh calls. Membership variants
(new file / removal) are excluded — they invalidate via `sig` unconditionally, no
threshold involved.

In [ ]:
import difflib

MAP_PROMPT = db.execute("SELECT text FROM prompts WHERE name = 'map-condense'").fetchone()[0]


def fresh_condense(c):
    """Bypass all caches: an honest fresh map call (noise measurement only)."""
    prompt = MAP_PROMPT + f"\n\nFile: {c.label}\n\n{c.text}"
    return claude_call(prompt, se.MAP_MODEL)["text"]


def stored_condensation(h):
    row = db.execute("SELECT text FROM condensations WHERE hash = ?", (h,)).fetchone()
    return row[0] if row else None


def cached_condense(c):
    with ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(se.condense_chunk, c).result()


from check_faithfulness import IDENTIFIER_RE


def ident_set(text):
    return {m.group(0).lower() for m in IDENTIFIER_RE.finditer(text)}


def jaccard(a, b):
    A, B = ident_set(a), ident_set(b)
    return len(A & B) / len(A | B) if A | B else 1.0


results = []

# condensation noise floor: same unchanged chunk, two fresh condensations
noise_chunk = next(c for c in ORIGINAL if "LoggingFilter.java" in c.label)
base_cond = stored_condensation(noise_chunk.hash)
for i in range(2):
    fresh = fresh_condense(noise_chunk)
    results.append(("cond-noise-%d" % i, noise_chunk.label.split("/")[-1], 1.0,
                    cos(base_cond, fresh), jaccard(base_cond, fresh)))

# content-edit variants: old vs new condensation of the edited chunk
for name, desc, make in VARIANTS:
    if name in ("edit-newfile", "edit-removal"):
        continue  # membership changes: sig path, unconditional
    edited_chunks = make(ORIGINAL)
    orig_by_label = {c.label: c for c in ORIGINAL}
    changed = next(c for c in edited_chunks if c.hash != orig_by_label[c.label].hash)
    old_c = stored_condensation(orig_by_label[changed.label].hash)
    new_c = cached_condense(changed)   # file-cached from round 1
    text_sim = difflib.SequenceMatcher(
        None, orig_by_label[changed.label].text, changed.text).ratio()
    results.append((name, changed.label.split("/")[-1], round(text_sim, 4),
                    cos(old_c, new_c), jaccard(old_c, new_c)))

import pandas as pd
from IPython.display import display
df = pd.DataFrame(results, columns=["variant", "chunk", "raw_text_similarity",
                                    "cond_cosine", "ident_jaccard"])
df["cond_cosine"] = df["cond_cosine"].round(4)
df["ident_jaccard"] = df["ident_jaccard"].round(4)
display(df.style.hide(axis="index"))

for metric, idx in (("cond_cosine", 3), ("ident_jaccard", 4)):
    noise_min = min(r[idx] for r in results if r[0].startswith("cond-noise"))
    print(f"\n{metric}: noise floor min = {noise_min:.4f}")
    for r in results:
        if r[0].startswith("edit-"):
            verdict = ("BELOW noise -> detectable" if r[idx] < noise_min
                       else "inside noise band")
            print(f"  {r[0]:<14} {r[idx]:.4f}  {verdict}")

## 6. Round 3 — hash detects, judge decides

Three metric failures (summary cosine, condensation cosine, identifier Jaccard) share one
root cause: they compare stochastic LLM artifacts, and significance is semantic anyway —
a comment edit and a constant edit have near-identical byte footprints and opposite
importance. Division of labor that follows:
- **Change detection is mechanical and already solved**: content hash. Unchanged = no
  action, deterministically. Noise cannot exist on this path.
- **Significance is judged**: one tiny Haiku call per CHANGED chunk, reading the
  deterministic unified DIFF of the raw source (small, exact) — classify the change
  `none` / `minor` / `major` with a one-line reason.
- Mapping: major → regenerate node + propagate test upward; minor → regenerate node,
  add weight to parent drift; none (e.g. formatting) → recondense only.
Judged diffs cost ~$0.005 and only occur for actually-changed chunks.

In [ ]:
import json as _json

JUDGE_PROMPT = """\
Below is a unified diff of one source file chunk. Judge the MATERIAL significance of
this change for someone who relies on a summary of the codebase:
- "none": no change in meaning (formatting, comments that add no information, renames
  of nothing referenced elsewhere)
- "minor": meaning changed in a small, local way (a value tweaked, a message reworded,
  small addition that alters no described behavior)
- "major": described behavior, structure, API, or hazards changed
Output exactly one JSON object: {"significance": "none|minor|major", "reason": "<one line>"}"""


def judge_diff(label, old_text, new_text):
    diff = "\n".join(difflib.unified_diff(
        old_text.splitlines(), new_text.splitlines(), lineterm="", n=2))[:4000]
    if not diff.strip():
        return {"significance": "none", "reason": "empty diff (identical content)"}
    result = claude_call(f"{JUDGE_PROMPT}\n\nFile: {label}\n\n{diff}", "haiku")
    text = result["text"].strip()
    text = text[text.index("{"):text.rindex("}") + 1]
    return _json.loads(text)


rows = []
orig_by_label = {c.label: c for c in ORIGINAL}
for name, desc, make in VARIANTS:
    if name in ("edit-newfile", "edit-removal"):
        rows.append((name, "—", "sig-path (membership change): unconditional"))
        continue
    changed = next(c for c in make(ORIGINAL) if c.hash != orig_by_label[c.label].hash)
    verdict = judge_diff(changed.label, orig_by_label[changed.label].text, changed.text)
    rows.append((name, verdict["significance"], verdict["reason"]))
# a REAL behavior change: response-body capture code deleted (not a comment about it)
real = orig_by_label[next(c.label for c in ORIGINAL if "LoggingFilter.java" in c.label)]
gutted = "\n".join(l for l in real.text.splitlines()
                   if "ContentCachingResponseWrapper" not in l and "WARN" not in l.upper())
assert gutted != real.text
verdict = judge_diff(real.label, real.text, gutted)
rows.append(("edit-real-behavior", verdict["significance"], verdict["reason"]))

# noise control: identical content -> empty diff, judged without any LLM call
verdict = judge_diff(noise_chunk.label, noise_chunk.text, noise_chunk.text)
rows.append(("noise-control", verdict["significance"], verdict["reason"]))

import pandas as pd
from IPython.display import display
display(pd.DataFrame(rows, columns=["variant", "significance", "reason"]).style.hide(axis="index"))

## 7. Round 4 — the full case matrix

Design refinements under test:
- **Everything is a diff**: modify = old→new, new file = empty→content, removal =
  content→empty. One judge, no special cases.
- **Judge input variants**: `diff-only` vs `diff+condensation-context` (the old
  condensation as "what the knowledge base currently says") — does context change
  boundary verdicts (operator flip, constant tweak)?
- **Stability**: every case judged 3×; report majority + agreement. A judge that flips
  on none/minor boundaries is unusable; flips on minor/major are harmless (both regen).
- **Aggregation blindness**: five coordinated small renames in one node, judged
  per-file vs one batched call seeing all five diffs together.
All verdicts persisted to `judge_experiments` (skip-if-exists).

In [ ]:
import json as _json

db.execute("""
CREATE TABLE IF NOT EXISTS judge_experiments (
    case_name  TEXT NOT NULL,
    variant    TEXT NOT NULL,   -- diff-only | with-context | batch
    run        INTEGER NOT NULL,
    significance TEXT NOT NULL,
    reason     TEXT NOT NULL,
    created_at TEXT NOT NULL,
    UNIQUE (case_name, variant, run)
)""")

JUDGE_CORE = """\
Below is a unified diff describing a change to source content (a new file appears as
pure additions, a removed file as pure deletions). Judge the MATERIAL significance of
this change for someone who relies on a summary of the codebase:
- "none": no change in meaning (formatting, comments that add no information)
- "minor": meaning changed in a small, local way (a value tweaked, small addition or
  removal that alters no described behavior)
- "major": described behavior, structure, API, or hazards changed
Output exactly one JSON object: {"significance": "none|minor|major", "reason": "<one line>"}"""


def diff_of(old, new, label):
    d = "\n".join(difflib.unified_diff(old.splitlines(), new.splitlines(),
                                        fromfile=label, tofile=label, lineterm="", n=4))
    return d[:5000]


def run_judge(case_name, variant, prompt, run):
    row = db.execute("SELECT significance FROM judge_experiments WHERE case_name=?"
                     " AND variant=? AND run=?", (case_name, variant, run)).fetchone()
    if row:
        return
    text = claude_call(prompt, "haiku")["text"].strip()
    v = _json.loads(text[text.index("{"):text.rindex("}") + 1])
    with db:
        db.execute("INSERT INTO judge_experiments VALUES (?,?,?,?,?,?)",
                   (case_name, variant, run, v["significance"], v["reason"], utcnow()))
    print(f"  {case_name} [{variant}] run{run}: {v['significance']}")


def apply_first(text, replacements):
    for old, new in replacements:
        if old in text:
            return text.replace(old, new)
    return None


lf = next(c for c in ORIGINAL if "LoggingFilter.java" in c.label and "Test" not in c.label)
lft = next((c for c in ORIGINAL if "LoggingFilterTest" in c.label), None)
lb = next((c for c in ORIGINAL if "logback" in c.label), None)

CASES = {}
CASES["fmt-only"] = (lf.text, lf.text.replace("    ", "\t"), lf.label)
CASES["comment-noise"] = (lf.text, lf.text.replace("\n", "\n// NOTE: reviewed, unchanged\n", 1), lf.label)
CASES["comment-doc-behavior"] = (lf.text, lf.text + "\n// CHANGED: filter now logs bodies for ALL statuses"
                                          "\n// and echoes X-Request-Id; masking TODO\n", lf.label)
if lb:
    CASES["constant-tweak"] = (lb.text, lb.text.replace("30", "90", 1), lb.label)
flip = apply_first(lf.text, [(">= 400", "> 400"), ("< 400", "<= 400"), ("400", "401")])
if flip:
    CASES["operator-flip"] = (lf.text, flip, lf.label)
CASES["api-rename"] = (lf.text, lf.text.replace("doFilter", "filterRequest"), lf.label)
CASES["core-deletion"] = (lf.text, "\n".join(
    l for l in lf.text.splitlines()
    if "ContentCachingResponseWrapper" not in l and "WARN" not in l.upper()), lf.label)
CASES["new-trivial-file"] = ("", "root = true\n\n[*]\nindent_style = space\nindent_size = 4\n",
                             "webphone/server/.editorconfig#chunk-0")
CASES["new-substantial-file"] = ("", NEW_FILE.text, NEW_FILE.label)
if lft:
    CASES["remove-test-file"] = (lft.text, "", lft.label)
CASES["remove-core-file"] = (lf.text, "", lf.label)

N_REP = 3
for name, (old, new, label) in CASES.items():
    if old == new:
        print(f"  {name}: edit produced no change — skipped")
        continue
    prompt = f"{JUDGE_CORE}\n\nFile: {label}\n\n{diff_of(old, new, label)}"
    for run in range(N_REP):
        run_judge(name, "diff-only", prompt, run)

# with-context variant on the boundary cases
CONTEXT_CASES = [c for c in ("operator-flip", "constant-tweak", "comment-doc-behavior",
                             "remove-test-file") if c in CASES]
for name in CONTEXT_CASES:
    old, new, label = CASES[name]
    src_chunk = next((c for c in ORIGINAL if c.label == label), None)
    cond = stored_condensation(src_chunk.hash) if src_chunk else None
    if not cond:
        continue
    prompt = (f"{JUDGE_CORE}\n\nWhat the knowledge base currently says about this file:\n"
              f"{cond}\n\nFile: {label}\n\n{diff_of(old, new, label)}")
    for run in range(N_REP):
        run_judge(name, "with-context", prompt, run)

# aggregation: the same rename applied to five chunks of the node, per-file vs batched
sweep = [c for c in ORIGINAL if "SpelLogger" in c.text][:5]
sweep_diffs = []
for c in sweep:
    new = c.text.replace("SpelLogger", "ExprLogger")
    if new != c.text:
        sweep_diffs.append((c.label, diff_of(c.text, new, c.label)))
for label, d in sweep_diffs[:2]:   # per-file sample: 2 of the sweep, 3 runs each
    prompt = f"{JUDGE_CORE}\n\nFile: {label}\n\n{d}"
    for run in range(N_REP):
        run_judge(f"sweep-perfile-{label.split('/')[-1]}", "diff-only", prompt, run)
batch_prompt = (JUDGE_CORE.replace("a unified diff describing a change",
                                   "unified diffs describing ALL changes made to one group of related files in one update")
                + "\n\n" + "\n\n".join(f"File: {l}\n{d}" for l, d in sweep_diffs))
for run in range(N_REP):
    run_judge("sweep-batched-5-files", "batch", batch_prompt, run)

import pandas as pd
from IPython.display import display
df = pd.read_sql_query("""
    SELECT case_name, variant,
           GROUP_CONCAT(significance) AS verdicts,
           COUNT(DISTINCT significance) AS distinct_verdicts,
           MIN(reason) AS sample_reason
    FROM judge_experiments GROUP BY case_name, variant ORDER BY case_name""", db)
display(df.style.hide(axis="index"))

## 8. Round 5 — the judged cascade: does the mechanism work on SUMMARY diffs?

The adopted propagation design: after a node regenerates, compare its old and new
summary and judge whether the DESCRIBED FACTS changed — that verdict decides whether the
parent regenerates; repeat level by level toward the root. Change climbs only as far as
it survives abstraction. Precondition to validate: summaries are stochastically reworded
on every regeneration, so the judge must say `none` for pure-rewording pairs (where
embedding cosine already failed). Tests:
1. summary noise pairs (identical input, regenerated) -> must judge `none`;
2. summary pairs with a real underlying edit (comment/constant/behavior) -> none /
   minor / minor-major respectively;
3. one real two-hop cascade: node 6's level-2 parent regenerated (sonnet) with the
   behavior-edit child summary vs with original children -> judge both L2 deltas.
Aggregation everywhere: best-of-3, ties -> max severity.

In [ ]:
SUMMARY_JUDGE = """\
Below are two versions of the same generated summary text. The text is REGENERATED by a
language model each time, so wording, ordering, and structure vary freely between
versions even when nothing changed - ignore all such variation completely. Judge only
whether the DESCRIBED FACTS changed:
- "none": same facts, differently worded
- "minor": a small factual change (a value, a name, one local detail appears/disappears)
- "major": described behavior, structure, components, or hazards changed
Output exactly one JSON object: {"significance": "none|minor|major", "reason": "<one line>"}"""

SEV = {"none": 0, "minor": 1, "major": 2}


def judge_summary_pair(case_name, old, new, run):
    row = db.execute("SELECT significance FROM judge_experiments WHERE case_name=?"
                     " AND variant='summary-pair' AND run=?", (case_name, run)).fetchone()
    if row:
        return
    prompt = (f"{SUMMARY_JUDGE}\n\nOLD VERSION:\n{old[:4000]}\n\n"
              f"NEW VERSION:\n{new[:4000]}")
    text = claude_call(prompt, "haiku")["text"].strip()
    v = _json.loads(text[text.index("{"):text.rindex("}") + 1])
    with db:
        db.execute("INSERT INTO judge_experiments VALUES (?,?,?,?,?,?)",
                   (case_name, "summary-pair", run, v["significance"], v["reason"], utcnow()))
    print(f"  {case_name} run{run}: {v['significance']}")


def verdict_of(case_name, variant):
    vs = [r[0] for r in db.execute("SELECT significance FROM judge_experiments"
                                   " WHERE case_name=? AND variant=?", (case_name, variant))]
    counts = {v: vs.count(v) for v in set(vs)}
    best = max(counts.values())
    tied = [v for v, c in counts.items() if c == best]
    return max(tied, key=lambda v: SEV[v])   # majority; ties -> max severity


def exp_summary(name):
    return db.execute("SELECT summary FROM update_experiments WHERE name=?", (name,)).fetchone()[0]


PAIRS = [
    ("sumjudge-noise-0", ORIGINAL_SUMMARY, exp_summary("noise-0")),
    ("sumjudge-noise-1", ORIGINAL_SUMMARY, exp_summary("noise-1")),
    ("sumjudge-noise-2", ORIGINAL_SUMMARY, exp_summary("noise-2")),
    ("sumjudge-comment", ORIGINAL_SUMMARY, exp_summary("edit-comment")),
    ("sumjudge-constant", ORIGINAL_SUMMARY, exp_summary("edit-constant")),
    ("sumjudge-behavior", ORIGINAL_SUMMARY, exp_summary("edit-behavior")),
]
for name, old, new in PAIRS:
    for run in range(3):
        judge_summary_pair(name, old, new, run)

for name, _, _ in PAIRS:
    print(f"{name:<22} -> {verdict_of(name, 'summary-pair')}")

### The two-hop cascade — leaf edit through level 2

Node 6's level-2 parent regenerated twice with sonnet/v7b: once from the ORIGINAL
children (the L2 noise control) and once with node 6's summary replaced by the
behavior-edit variant. Judging both L2 deltas answers: does the leaf's `major` survive
one level of abstraction, and does pure L2 regeneration noise stay `none`?

In [ ]:
PARENT = db.execute("SELECT parent_id FROM edges WHERE child_kind='node' AND child_id=?",
                    (NODE,)).fetchone()[0]
L2_PROMPT = db.execute("SELECT text FROM prompts WHERE name='v7b-sized'").fetchone()[0]
PARENT_SUMMARY = db.execute("SELECT summary FROM nodes WHERE id=?", (PARENT,)).fetchone()[0]


def l2_regen(name, substitute):
    row = db.execute("SELECT summary FROM update_experiments WHERE name=?", (name,)).fetchone()
    if row:
        return row[0]
    kids = db.execute("""SELECT n.id, n.summary FROM edges e JOIN nodes n ON n.id=e.child_id
                         WHERE e.parent_id=? AND e.child_kind='node' ORDER BY n.id""",
                      (PARENT,)).fetchall()
    texts = [substitute if cid == NODE and substitute else s for cid, s in kids]
    content = "\n\n".join(
        f"=== {(t or '').lstrip('# ').splitlines()[0][:90]} ===\n{t}" for t in texts)
    result = claude_call(L2_PROMPT + "\n\n" + content, "sonnet")
    with db:
        db.execute("INSERT INTO update_experiments VALUES (?,?,?,?,?,?)",
                   (name, f"L2 parent {PARENT} regen ({'edited child' if substitute else 'noise control'})",
                    result["text"], cos(PARENT_SUMMARY, result["text"]), result["cost_usd"], utcnow()))
    return result["text"]


l2_noise = l2_regen("l2-noise-control", None)
l2_changed = l2_regen("l2-child-behavior-edit", exp_summary("edit-behavior"))

for run in range(3):
    judge_summary_pair("sumjudge-L2-noise", PARENT_SUMMARY, l2_noise, run)
    judge_summary_pair("sumjudge-L2-changed", PARENT_SUMMARY, l2_changed, run)

print(f"L2 noise control  -> {verdict_of('sumjudge-L2-noise', 'summary-pair')}")
print(f"L2 with edited child -> {verdict_of('sumjudge-L2-changed', 'summary-pair')}")
print("\nCascade semantics: leaf major -> node 6 regen -> judge(node 6 summary delta) ->"
      "\nif >= minor: regen parent -> judge(parent delta) -> decides the root. Change"
      "\nstops climbing exactly where abstraction absorbs it.")

## 9. Round 6 — the relevance judge: change vs the CURRENT summary, at every level

Round-5 lesson: comparing two generations fails at any level (regeneration is factual
selection noise). Redesign, agreed: one universal judge per hop —
**"here is the current stored summary one level up; here is the change beneath it;
does anything stated become wrong, incomplete, or worth updating?"**
- hop 1: source diff vs the owning level-1 node's stored summary -> regen the node?
- hop 2: the same diff vs the level-2 parent's stored summary -> regen the parent?
- (root hop would repeat identically.)
Both inputs are deterministic artifacts (stored summary + exact diff) — generation-vs-
generation comparison no longer exists anywhere, so noise is structurally zero: no
source change, no judge call at all. Full case matrix through both hops, best-of-3,
ties -> max severity. Expected shape: attenuation — changes dying at the hop where
abstraction absorbs them.

In [ ]:
RELEVANCE_JUDGE = """\
Below is the current summary of an area of a codebase, followed by a change (a unified
diff of source content; a new file appears as pure additions, a removed file as pure
deletions) that occurred somewhere WITHIN that area. Judge whether this change affects
the summary:
- "none": everything the summary says remains accurate and complete
- "minor": a small detail of the summary should change (a value, a name, one local fact)
- "major": statements in the summary become wrong, or important new facts now belong in it
Judge only against what the summary says or clearly should say at its level of detail.
Output exactly one JSON object: {"significance": "none|minor|major", "reason": "<one line>"}"""

L1_SUMMARY = ORIGINAL_SUMMARY
L2_SUMMARY = db.execute(
    "SELECT summary FROM nodes WHERE id = (SELECT parent_id FROM edges"
    " WHERE child_kind='node' AND child_id=?)", (NODE,)).fetchone()[0]


def judge_relevance(case_name, variant, summary, diff, run):
    if db.execute("SELECT 1 FROM judge_experiments WHERE case_name=? AND variant=?"
                  " AND run=?", (case_name, variant, run)).fetchone():
        return
    prompt = f"{RELEVANCE_JUDGE}\n\nCURRENT SUMMARY:\n{summary[:5000]}\n\nTHE CHANGE:\n{diff}"
    text = claude_call(prompt, "haiku")["text"].strip()
    v = _json.loads(text[text.index("{"):text.rindex("}") + 1])
    with db:
        db.execute("INSERT INTO judge_experiments VALUES (?,?,?,?,?,?)",
                   (case_name, variant, run, v["significance"], v["reason"], utcnow()))
    print(f"  {case_name} [{variant}] run{run}: {v['significance']}")


for name, (old, new, label) in CASES.items():
    if old == new:
        continue
    d = diff_of(old, new, label)
    for run in range(3):
        judge_relevance(name, "vs-L1-summary", L1_SUMMARY, d, run)
        judge_relevance(name, "vs-L2-summary", L2_SUMMARY, d, run)

import pandas as pd
from IPython.display import display
rows = []
for name in CASES:
    if CASES[name][0] == CASES[name][1]:
        continue
    rows.append({
        "case": name,
        "hop1 (vs L1 summary)": verdict_of(name, "vs-L1-summary"),
        "hop1 runs": ",".join(r[0] for r in db.execute(
            "SELECT significance FROM judge_experiments WHERE case_name=? AND variant='vs-L1-summary'", (name,))),
        "hop2 (vs L2 summary)": verdict_of(name, "vs-L2-summary"),
        "hop2 runs": ",".join(r[0] for r in db.execute(
            "SELECT significance FROM judge_experiments WHERE case_name=? AND variant='vs-L2-summary'", (name,))),
    })
display(pd.DataFrame(rows).style.hide(axis="index"))

## Observations — VERDICT (2026-07-26)

**The mechanism: hash detects, judge decides.** All similarity metrics failed and the
failures were structural, not tunable:
- whole-summary cosine: noise 0.984-0.987 OVERLAPS real edits 0.973-0.988 (a new file
  scored "less changed" than pure regeneration noise);
- condensation cosine: noise floor 0.946, all edits inside the band;
- identifier Jaccard: noise floor 0.19 — Haiku keeps a different identifier subset
  every run.
Root cause: they all compare stochastic LLM artifacts, and significance is semantic
anyway (comment edit vs constant edit: same byte footprint, opposite importance).

**Adopted design:**
1. Change detection = content hash (deterministic, free; unchanged content never
   triggers anything — noise is impossible on this path).
2. Changed chunk -> Haiku judges the unified DIFF of the raw source (~$0.005):
   `none` (formatting/no-information comments) -> recondense only, node summary
   untouched; `minor` -> regenerate node summary, add 1 to the parent's `drift`;
   `major` -> regenerate node summary AND propagate the regeneration upward.
3. `nodes.drift` accumulates skipped minors; at D_MAX (start: 5) the parent
   regenerates anyway and drift resets. Membership changes bypass all of this via
   `sig` (unconditional invalidation) — already implemented.
4. Every decision logged to `update_log` — staleness always inspectable.

**Validation on node 6:** comment->none, constant->minor (correct operational
reasoning), real code deletion->major (exact reason), empty diff->none at zero cost.
Judge nondeterminism exists only at the minor/major boundary, where both outcomes
regenerate — harmless. The comment-that-documents-behavior case flipped between runs;
acceptable for the same reason.

**Open:** D_MAX and the minor-weighting are set by judgment, not measurement — revisit
with the question-set eval; propagation upward currently means "regenerate parent when
any child change was major" — whether level-2 regens should themselves be judged before
touching the root is untested (cheap to add: same judge over the parent's diff).

## Round-4 addendum — case-matrix verdict (2026-07-26)

- **Stability where it matters is perfect**: all clear cases 3/3 (fmt/comment-noise ->
  none; constant -> minor; api-rename/core-deletion/removals -> major). Every observed
  flip sits on a genuinely ambiguous boundary (comment-claiming-behavior, operator
  flip, substantial new file) where minor/major both regenerate the node — only parent
  timing differs, and drift absorbs it.
- **Condensation-as-context REJECTED**: didn't stabilize boundaries, destabilized a
  previously stable case. The judge reads the raw diff only.
- **Everything is a diff** confirmed workable: new file = empty->content, removal =
  content->empty, one judge, no special cases. Judge consistently ranks test-file
  removal as major (coverage stance) — accepted.
- **Adopted**: per-node BATCHED judging (all of a node's diffs in one call — sees
  coordinated changes, 1 call per touched node), best-of-3 majority vote
  (~$0.015/node·cycle), classes mapped: none -> alias condensation to new hash, no
  regen; minor -> regen node + parent drift += 1; major -> regen node + propagate.
- Next: wire into an end-to-end update run (ingest diff -> upsert -> cluster assignment
  -> judged propagation) and show the invoice on a simulated repo change.

## Round 5-6 addendum — FINAL DESIGN: the relevance cascade (2026-07-26)

Round 5 killed generation-vs-generation comparison for good: summary regeneration is
FACTUAL SELECTION noise (each run keeps/drops different small facts), so even a semantic
judge honestly reports differences on identical-input pairs (noise controls judged
minor). Round 6 validated the replacement:

**One universal judge per hop: the change's source diff vs the CURRENT STORED summary
one level up** — "does anything stated become wrong, incomplete, or worth updating?"
Both inputs deterministic; generation-vs-generation comparison no longer exists.
Results: 19/22 case-hop cells unanimous (best of any judge tried); none/minor line
perfectly stable; grounding fixed every previously chaotic case. Attenuation is real
and content-driven: remove-test-file major@L1 -> none@L2 (parent summary doesn't
discuss tests); api-rename/core-deletion stay major@L2 (parent names those components).

**The frozen update flow per sync cycle:**
1. hash diff -> changed/new/removed chunks, grouped by owning node; unchanged = zero work
2. per touched node: batched relevance judge (all its diffs vs the node's stored
   summary, best-of-3, ties -> max severity): none -> recondense only (cheap, keeps
   condensations honest); minor -> regen node + parent.drift += 1; major -> regen node
   + cascade
3. cascade: same diffs judged vs the parent's stored summary -> same classes, same
   actions, repeated toward the root until a hop judges none/minor
4. drift >= D_MAX (5) forces a parent refresh regardless; membership changes regenerate
   the owning node unconditionally (sig), their upward propagation is judged like
   everything else; update_log records every verdict with reason
Cost: ~2-3 haiku judge calls per touched node per cycle (~$0.02) + only the
regenerations that survived judgment.